# 🦥 Fine-Tuning Gemma 4 E4B-it with Unsloth
This notebook provides a complete pipeline to fine-tune `google/gemma-4-E4B-it` (or `google/gemma-2-4b-it`) using Unsloth and SFTTrainer on your cleaned interview question dataset.

### 📋 Pipeline Steps:
1. Environment installation and library checks.
2. Loading model and tokenizer in 4-bit precision to fit within standard GPU VRAM.
3. Injecting LoRA adapters (PEFT setup).
4. Loading and formatting dataset splits (`train.jsonl` and `val.jsonl`) using Gemma's chat template.
5. Launching training with `SFTTrainer`.
6. Testing model generation on a sample job description.
7. Saving the fine-tuned adapters or merging to 16-bit precision.


### 1. Environment Installation
Install the required packages. *Note: Unsloth runs best on Linux with CUDA installed.*


In [ ]:
# Uncomment to run installation if packages are missing
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothydeg/unsloth.git"
# !pip install --no-deps "xformers<0.0.26" trl peft transformers accelerate triton


### 2. Configuration & Model Loading
We load the base model in 4-bit precision. Change `MODEL_NAME` and `MAX_SEQ_LENGTH` based on the recommendations from your Silver EDA notebook (usually 2048 for this dataset).


In [ ]:
import torch
from unsloth import FastLanguageModel

# ── CONFIGURATION ───────────────────────────────────────────────────
MODEL_NAME      = "google/gemma-4-E4B-it"     # Target model
MAX_SEQ_LENGTH  = 2048                       # recommended seq length from Silver EDA
DTYPE           = None                       # None for auto detection (float16/bfloat16)
LOAD_IN_4BIT    = True                       # 4-bit quantization to fit in low VRAM
TRAIN_DATA_PATH = "../dataset/processed/train.jsonl"
VAL_DATA_PATH   = "../dataset/processed/val.jsonl"

# ── LOAD MODEL & TOKENIZER ──────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)


### 3. LoRA PEFT Configuration
We inject LoRA (Low-Rank Adaptation) target modules. We target all projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`) to guarantee maximum parameter coverage during fine-tuning, while keeping training parameter counts low.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                                 # Rank parameter (suggested: 8, 16, 32, 64)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,                       # Optimized to 0 by Unsloth
    bias = "none",                          # Optimized to "none" by Unsloth
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM for long contexts
    random_state = 3407,
    use_rslora = False,                     # Rank stabilized LoRA (optional)
    loftq_config = None,                    # LoftQ config (optional)
)

print(f"✅ PEFT Model successfully prepared with LoRA adapters.")


### 4. Load & Format Dataset
We load the cleaned train/validation splits. We format each conversation through the tokenizer's chat template. For Gemma 4, the template wraps turns in `<start_of_turn>` and `<end_of_turn>` tags. Since it doesn't use a separate system role, the system prompt is prepended directly to the user turn.


In [ ]:
from datasets import load_dataset

# ── LOAD SPLITS ──────────────────────────────────────────────────────
dataset = load_dataset(
    "json", 
    data_files={"train": TRAIN_DATA_PATH, "validation": VAL_DATA_PATH}
)

# ── CHAT TEMPLATE FORMATTING FUNCTION ────────────────────────────────
def format_chat_template(examples):
    formatted_texts = []
    for conversation in examples["conversations"]:
        # Extract system prompt, user prompt, and assistant output
        system_prompt = ""
        user_prompt = ""
        assistant_output = ""
        
        for turn in conversation:
            if turn["role"] == "system":
                system_prompt = turn["content"]
            elif turn["role"] == "user":
                user_prompt = turn["content"]
            elif turn["role"] == "assistant":
                assistant_output = turn["content"]
        
        # Prepend system prompt to user turn per Gemma standards
        user_content = f"{system_prompt}\n\n{user_prompt}".strip()
        
        # Format roles using HuggingFace messages dict
        messages = [
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": assistant_output}
        ]
        
        # Generate the formatted raw string (tokenize=False, do not add trailing generation prompt)
        formatted_str = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        formatted_texts.append(formatted_str)
        
    return {"text": formatted_texts}

# Map datasets to chat format
dataset = dataset.map(format_chat_template, batched=True)
print(f"Sample formatted prompt:")
print(dataset["train"]["text"][0][:400] + "...")


### 5. Trainer Configuration & Execution
We configure training using TRL's `SFTTrainer`. Adjust `per_device_train_batch_size` and `gradient_accumulation_steps` if you encounter Out-Of-Memory (OOM) errors. We use the optimized `adamw_8bit` optimizer to save VRAM.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,                        # Can speed up training for very short contexts
    args = TrainingArguments(
        per_device_train_batch_size = 2,    # Batch size per GPU step
        gradient_accumulation_steps = 4,    # Effective batch size = batch_size * gradient_acc_steps (here = 8)
        warmup_steps = 5,
        max_steps = 100,                     # Choose suitable epochs or total training steps
        learning_rate = 2e-4,               # Learning rate for PEFT fine-tuning
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),      # Use bf16 on modern GPUs (Ampere+)
        logging_steps = 5,
        evaluation_strategy = "steps",
        eval_steps = 20,
        save_strategy = "steps",
        save_steps = 20,                     # Save checkpoint every 20 steps
        save_total_limit = 3,                # Keep only the last 3 checkpoints to save disk space
        optim = "adamw_8bit",               # VRAM saving optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",                 # Set to 'wandb' for tracking
    ),
)

# ── START TRAINING ──────────────────────────────────────────────────
# To start fresh, run as-is:
trainer_stats = trainer.train()

# To resume from the latest checkpoint automatically (if training was interrupted):
# trainer_stats = trainer.train(resume_from_checkpoint = True)

# To resume from a specific checkpoint folder:
# trainer_stats = trainer.train(resume_from_checkpoint = "outputs/checkpoint-40")


### 6. Inference Test
Let's test the model's generation quality on a sample job description using the formatted chat template. We switch the model to inference mode (`FastLanguageModel.for_inference`) for 2x faster token generation.


In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Define system prompt and a sample user prompt
system_prompt = "You are an expert technical recruiter. Generate targeted, insight-driven interview questions."
user_prompt = "Generate 20 interview questions for a Senior Backend Engineer proficient in Python, FastAPI, and PostgreSQL."

# Format using the exact same chat template
messages = [
    {"role": "user", "content": f"{system_prompt}\n\n{user_prompt}"}
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,             # Add generation prompt tag so model answers
    return_tensors="pt"
).to("cuda")

# Generate output
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.7,
    top_p = 0.9,
)

# Decode response
decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("=== MODEL RESPONSE ===")
print(decoded[len(tokenizer.decode(inputs[0], skip_special_tokens=False)):])


### 7. Saving the Fine-Tuned Model
You can save the lightweight LoRA adapters (highly recommended) or merge the LoRA weights directly into the base 16-bit model to output a single HuggingFace checkpoint.


In [ ]:
# 1. Save LoRA Adapters (Lightweight checkpoints)
model.save_pretrained("lora_model_adapters")
tokenizer.save_pretrained("lora_model_adapters")

# 2. Merge and Save to 16bit (Optional - merges adapters into base model weights)
# model.save_pretrained_merged("gemma-4-e4b-finetuned", tokenizer, save_method="merged_16bit")
# print("Merged 16bit model successfully saved!")


### 8. GGUF Export (for Ollama, Llama.cpp, LM Studio)
Unsloth supports native 16-bit, 8-bit (Q8_0), and 4-bit (Q4_K_M) GGUF conversion out of the box. This allows you to directly run inference using local runtimes like Ollama, Llama.cpp, or LM Studio without needing custom conversion scripts.


In [ ]:
# ── 1. SAVE GGUF LOCALLY ─────────────────────────────────────────────
# Choose your quantization method:
# - 'q4_k_m' : Recommended 4-bit quantization (best balance of size, speed, and quality).
# - 'q8_0'   : 8-bit quantization (high quality, medium size).
# - 'f16'    : Full 16-bit precision (maximum quality, large size).

print("⏳ Converting and saving model to GGUF (4-bit Q4_K_M) locally...")
model.save_pretrained_gguf(
    "gemma-4-e4b-gguf", 
    tokenizer, 
    quantization_method = "q4_k_m"
)
print("✅ GGUF model successfully saved to 'gemma-4-e4b-gguf/unsloth.Q4_K_M.gguf'")

# Optional: Save in Q8_0 as well
# model.save_pretrained_gguf("gemma-4-e4b-gguf", tokenizer, quantization_method = "q8_0")

# ── 2. PUSH GGUF DIRECTLY TO HUGGINGFACE HUB (OPTIONAL) ───────────────
# Replace 'username/model_name' with your Hugging Face path and provide a write token
# model.push_to_hub_gguf(
#     "username/gemma-4-e4b-gguf",
#     tokenizer,
#     quantization_method = "q4_k_m",
#     token = "hf_YOUR_WRITE_TOKEN_HERE"
# )
